In [1]:
#Step 1: Create the SQL notebook and build the database
from pathlib import Path
import sqlite3
import pandas as pd

project = Path.cwd().parent
processed_dir = project / 'data' / 'processed'
db_path = project / 'data' / 'payshift.db'

npci = pd.read_csv(processed_dir / 'npci_monthly_2024.csv')
rbi = pd.read_csv(processed_dir / 'rbi_bank_cards_2024.csv')

conn = sqlite3.connect(db_path)

npci.to_sql('npci_monthly', conn, if_exists='replace', index=False)
rbi.to_sql('rbi_bank_cards', conn, if_exists='replace', index=False)

pd.read_sql("SELECT name FROM sqlite_master WHERE type = 'table'", conn)

,name
0,npci_monthly
1,rbi_bank_cards


In [2]:
#Step 2: Your first analysis query

In [3]:
query = """
SELECT
    month,
    SUM(CASE WHEN system = 'UPI'  THEN volume_mn END) AS upi_mn,
    SUM(CASE WHEN system = 'IMPS' THEN volume_mn END) AS imps_mn,
    SUM(CASE WHEN system = 'NFS'  THEN volume_mn END) AS atm_nfs_mn
FROM npci_monthly
GROUP BY month
ORDER BY month;
"""

monthly_systems = pd.read_sql(query,conn)
monthly_systems

,month,upi_mn,imps_mn,atm_nfs_mn
0,2024-01-01,12203.02,508.79,313.73
1,2024-02-01,12102.67,534.64,304.52
2,2024-03-01,13440.00,580.64,328.77
3,2024-04-01,13303.99,550.37,312.76
4,2024-05-01,14035.84,557.70,307.71
5,2024-06-01,13885.14,516.75,302.55
6,2024-07-01,14435.55,490.28,298.74
7,2024-08-01,14963.05,453.34,304.34
8,2024-09-01,15041.75,429.94,292.89
9,2024-10-01,16584.97,466.82,329.11


In [4]:
#Step 3: Month-over-month (MoM) growth with a window function

In [5]:
query = """
WITH monthly AS (
    SELECT
        month,
        SUM(CASE WHEN system = 'UPI' THEN volume_mn END) AS upi_mn,
        SUM(CASE WHEN system = 'NFS' THEN volume_mn END) AS atm_nfs_mn
    FROM npci_monthly
    GROUP BY month
)
SELECT
    month,
    upi_mn,
    LAG(upi_mn) OVER (ORDER BY month) AS upi_prev_mn,
    ROUND((upi_mn - LAG(upi_mn) OVER (ORDER BY month)) * 100.0
          / LAG(upi_mn) OVER (ORDER BY month), 1) AS upi_mom_pct,
    atm_nfs_mn,
    ROUND((atm_nfs_mn - LAG(atm_nfs_mn) OVER (ORDER BY month)) * 100.0
          / LAG(atm_nfs_mn) OVER (ORDER BY month), 1) AS atm_mom_pct
FROM monthly
ORDER BY month;
"""

mom = pd.read_sql(query, conn)
mom

,month,upi_mn,upi_prev_mn,upi_mom_pct,atm_nfs_mn,atm_mom_pct
0,2024-01-01,12203.02,NaN,NaN,313.73,NaN
1,2024-02-01,12102.67,12203.02,-0.8,304.52,-2.9
2,2024-03-01,13440.00,12102.67,11.0,328.77,8.0
3,2024-04-01,13303.99,13440.00,-1.0,312.76,-4.9
4,2024-05-01,14035.84,13303.99,5.5,307.71,-1.6
5,2024-06-01,13885.14,14035.84,-1.1,302.55,-1.7
6,2024-07-01,14435.55,13885.14,4.0,298.74,-1.3
7,2024-08-01,14963.05,14435.55,3.7,304.34,1.9
8,2024-09-01,15041.75,14963.05,0.5,292.89,-3.8
9,2024-10-01,16584.97,15041.75,10.3,329.11,12.4


In [6]:
#Step 4: MoM growth on daily averages

In [7]:
query = """
WITH monthly AS (
    SELECT
        month,
        SUM(CASE WHEN system = 'UPI' THEN avg_daily_volume_mn END) AS upi_daily_mn,
        SUM(CASE WHEN system = 'NFS' THEN avg_daily_volume_mn END) AS atm_daily_mn
    FROM npci_monthly
    GROUP BY month
)
SELECT
    month,
    upi_daily_mn,
    ROUND((upi_daily_mn - LAG(upi_daily_mn) OVER (ORDER BY month)) * 100.0
          / LAG(upi_daily_mn) OVER (ORDER BY month), 1) AS upi_daily_mom_pct,
    atm_daily_mn,
    ROUND((atm_daily_mn - LAG(atm_daily_mn) OVER (ORDER BY month)) * 100.0
          / LAG(atm_daily_mn) OVER (ORDER BY month), 1) AS atm_daily_mom_pct
FROM monthly
ORDER BY month;
"""

mom_daily = pd.read_sql(query, conn)
mom_daily

,month,upi_daily_mn,upi_daily_mom_pct,atm_daily_mn,atm_daily_mom_pct
0,2024-01-01,393.65,NaN,10.12,NaN
1,2024-02-01,417.33,6.0,10.50,3.8
2,2024-03-01,433.55,3.9,10.61,1.0
3,2024-04-01,443.47,2.3,10.43,-1.7
4,2024-05-01,452.77,2.1,9.93,-4.8
5,2024-06-01,462.84,2.2,10.09,1.6
6,2024-07-01,465.66,0.6,9.64,-4.5
7,2024-08-01,482.68,3.7,9.82,1.9
8,2024-09-01,501.39,3.9,9.76,-0.6
9,2024-10-01,535.00,6.7,10.62,8.8


In [8]:
#Step 5: Q1 vs Q4, a fairer full-year comparison
#The question: Over 2024, which systems grew and which declined?
#We'll compare the average daily volume in Q1 (Jan–Mar) with Q4 (Oct–Dec) for every system.

In [9]:
query = """
WITH quarterly AS (
    SELECT
        system,
        CASE
            WHEN CAST(strftime('%m', month) AS INTEGER) <= 3  THEN 'Q1'
            WHEN CAST(strftime('%m', month) AS INTEGER) >= 10 THEN 'Q4'
        END AS quarter,
        avg_daily_volume_mn
    FROM npci_monthly
)
SELECT
    system,
    ROUND(AVG(CASE WHEN quarter = 'Q1' THEN avg_daily_volume_mn END), 2) AS q1_daily_mn,
    ROUND(AVG(CASE WHEN quarter = 'Q4' THEN avg_daily_volume_mn END), 2) AS q4_daily_mn,
    ROUND((AVG(CASE WHEN quarter = 'Q4' THEN avg_daily_volume_mn END)
         - AVG(CASE WHEN quarter = 'Q1' THEN avg_daily_volume_mn END)) * 100.0
         / AVG(CASE WHEN quarter = 'Q1' THEN avg_daily_volume_mn END), 1) AS q1_to_q4_pct
FROM quarterly
GROUP BY system
ORDER BY q1_to_q4_pct DESC;
"""

q_growth = pd.read_sql(query, conn)
q_growth

#Comparing 3-month averages is more reliable than comparing single months (January vs December), 
#because one unusual month can't distort the result.

,system,q1_daily_mn,q4_daily_mn,q1_to_q4_pct
0,UPI,414.84,530.25,27.8
1,AePS - BHIM Aadhaar Pay,0.06,0.07,11.1
2,AePS - Cash Withdrawal,2.98,3.30,10.8
3,NETC,10.92,11.80,8.1
4,NACH - APBS,8.93,9.46,6.0
5,NACH - Credit,4.69,4.58,-2.3
6,NFS,10.41,9.80,-5.9
7,AePS - Funds Transfer,0.00,0.00,-9.1
8,IMPS,17.86,14.30,-20.0


In [10]:
#UPI grew 27.8%, from about 415 Mn to 530 Mn transactions a day. By Q4, UPI handled about 54 times as many transactions as the ATM network.
#IMPS fell 20%, the clearest sign of substitution: instant bank transfers are moving to UPI.
#ATM (NFS) fell 5.9%, so cash use through the interbank ATM network is declining, slowly.
#AePS cash withdrawal rose 10.8%: rural cash demand is still growing, while urban ATM use falls.

In [11]:
#Step 6: Bank-level analysis, credit card market share

In [12]:
query = """
SELECT
    bank_name,
    bank_category,
    credit_cards,
    ROUND(credit_cards * 100.0 / SUM(credit_cards) OVER (), 2) AS market_share_pct,
    RANK() OVER (ORDER BY credit_cards DESC) AS card_rank
FROM rbi_bank_cards
WHERE month = '2024-12-01'
ORDER BY credit_cards DESC
LIMIT 10;
"""

cc_share = pd.read_sql(query, conn)
cc_share

,bank_name,bank_category,credit_cards,market_share_pct,card_rank
0,HDFC BANK LTD,Private Sector Banks,23145038,21.42,1
1,STATE BANK OF INDIA,Public Sector Banks,20246942,18.74,2
2,ICICI BANK LTD,Private Sector Banks,17901052,16.57,3
3,AXIS BANK LTD,Private Sector Banks,14796963,13.69,4
4,RBL BANK LTD,Private Sector Banks,5038440,4.66,5
5,KOTAK MAHINDRA BANK LTD,Private Sector Banks,5021613,4.65,6
6,IDFC FIRST BANK LTD,Private Sector Banks,3291800,3.05,7
7,INDUSIND BANK LTD,Private Sector Banks,3117373,2.88,8
8,BANK OF BARODA,Public Sector Banks,2937896,2.72,9
9,YES BANK LTD,Private Sector Banks,2407944,2.23,10


1. How concentrated is the market?

Top 4 banks (HDFC, SBI, ICICI, Axis): about 70% of all credit cards (21.42 + 18.74 + 16.57 + 13.69)
Top 5: about 75%
There's a sharp cliff after #4: Axis has 13.7%, but #5 (RBL) has only 4.7%. India's credit card market is effectively four big players plus a long tail.

HDFC's 23.1 Mn cards are 21.42% of the market, so India had about 108 million credit cards in December 2024 (23.1 ÷ 0.2142). Compare that with over a billion debit cards: credit cards are still a small, concentrated product.

2. Which categories appear?
8 of the top 10 are private banks. Only 2 are public sector banks: SBI at #2 and Bank of Baroda at #9.
No foreign banks and no small finance banks make the top 10.
This contrast is striking because public sector banks dominate debit cards (remember Bank of Baroda's 80 Mn debit cards). 
Public banks lead mass banking; private banks lead credit.

#Step 7: Market share by bank category
#The top 10 suggests private banks dominate credit cards. Let's confirm that across all 64 banks, 
#and compare credit with debit cards at the same time:

In [13]:
query = """
SELECT
    bank_category,
    COUNT(*) AS num_banks,
    SUM(credit_cards) AS credit_cards,
    ROUND(SUM(credit_cards) * 100.0 / SUM(SUM(credit_cards)) OVER (), 1) AS credit_share_pct,
    SUM(debit_cards) AS debit_cards,
    ROUND(SUM(debit_cards) * 100.0 / SUM(SUM(debit_cards)) OVER (), 1) AS debit_share_pct
FROM rbi_bank_cards
WHERE month = '2024-12-01'
GROUP BY bank_category
ORDER BY credit_cards DESC;
"""

category_share = pd.read_sql(query, conn)
category_share

,bank_category,num_banks,credit_cards,credit_share_pct,debit_cards,debit_share_pct
0,Private Sector Banks,21,76603727,70.9,247643996,25.0
1,Public Sector Banks,12,25761145,23.8,639019656,64.5
2,Foreign Banks,14,4593728,4.3,4564550,0.5
3,Small Finance Banks,11,1097305,1.0,30326184,3.1
4,Payment Banks,6,0,0.0,69393668,7.0


Public banks hold almost two-thirds of India's debit cards (639 Mn), but less than a quarter of its credit cards. Private banks are the reverse. In other words, public banks have the customers, while private banks have the credit business.

What each category tells you:

Public banks: about 25 debit cards for every credit card (639 ÷ 25.8). Their customer base is huge, likely helped by mass-account schemes that give every new account a debit card, but very few of those customers hold their credit cards.
Private banks: about 3 debit cards per credit card (247.6 ÷ 76.6). They're far better at converting customers into credit users.
Foreign banks: only 0.5% of debit cards but 4.3% of credit cards, so they hold almost as many credit cards as debit cards. That's a premium, urban niche.
Payment banks: zero credit cards, and that's correct, not a data error. Payment banks aren't allowed to lend under RBI rules, so they can't issue credit cards. Yet they hold 7% of debit cards. Knowing why a zero exists is exactly the kind of context to mention in interviews.
Overall, India has about 991 Mn debit cards vs 108 Mn credit cards, roughly 9 to 1.

Step 8: Credit card penetration by bank

In [14]:
query = """
SELECT
    bank_name,
    bank_category,
    debit_cards,
    credit_cards,
    ROUND(credit_cards * 100.0 / debit_cards, 1) AS cc_per_100_dc
FROM rbi_bank_cards
WHERE month = '2024-12-01'
  AND debit_cards >= 10000000
ORDER BY cc_per_100_dc ASC;
"""

penetration = pd.read_sql(query, conn)
penetration

,bank_name,bank_category,debit_cards,credit_cards,cc_per_100_dc
0,CENTRAL BANK OF INDIA,Public Sector Banks,28178425,0,0.0
1,UCO BANK,Public Sector Banks,12454245,0,0.0
2,FINO PAYMENTS BANK,Payment Banks,10036287,0,0.0
3,INDIA POST PAYMENTS BANK,Payment Banks,21861433,0,0.0
4,PAYTM PAYMENTS BANK,Payment Banks,30605276,0,0.0
5,BANK OF INDIA,Public Sector Banks,38747535,73696,0.2
6,BANK OF MAHARASHTRA,Public Sector Banks,14022969,31278,0.2
7,INDIAN OVERSEAS BANK,Public Sector Banks,20478463,65086,0.3
8,IDBI BANK LTD,Private Sector Banks,12107833,41023,0.3
9,INDIAN BANK,Public Sector Banks,33020751,282514,0.9


1. Some zeros are explained, others need checking.

The payment banks (Fino, India Post, Paytm) have zero by law. They can't lend, so they aren't an "opportunity". Exclude them from any recommendation.
Central Bank of India and UCO Bank report zero credit cards despite 28 Mn and 12 Mn debit cards. Before calling this an opportunity, an analyst would check why. They might offer cards through a partner that RBI counts elsewhere, or have left the business. Treat it as "needs verification", not a conclusion.

2. There's a clear public vs private divide.

Public banks: 0.2 to 3.4 credit cards per 100 debit cards
Private banks: 14.5 to 56.3. ICICI issues 56 credit cards for every 100 debit cards.
SBI (8.4) is the exception: the only public bank anywhere near private-bank levels. That's useful, because it shows a public bank can do this.

3. The biggest absolute opportunities. A low ratio matters most when the customer base is huge:

Union Bank: 60.6 Mn debit cards at just 0.9
Canara Bank: 59.4 Mn at 1.7
Bank of Baroda: 87.3 Mn at 3.4

These three banks serve over 200 Mn debit card customers but have issued very few credit cards.

Step 9: Size the opportunity

In [15]:
query = """
WITH sbi AS (
    SELECT credit_cards * 1.0 / debit_cards AS benchmark_ratio
    FROM rbi_bank_cards
    WHERE month = '2024-12-01'
      AND bank_name = 'STATE BANK OF INDIA'
)
SELECT
    r.bank_name,
    r.debit_cards,
    r.credit_cards,
    ROUND(r.credit_cards * 100.0 / r.debit_cards, 1) AS cc_per_100_dc,
    ROUND((r.debit_cards * s.benchmark_ratio - r.credit_cards) / 1000000.0, 2) AS extra_cards_mn
FROM rbi_bank_cards AS r
CROSS JOIN sbi AS s
WHERE r.month = '2024-12-01'
  AND r.bank_category = 'Public Sector Banks'
  AND r.bank_name <> 'STATE BANK OF INDIA'
ORDER BY extra_cards_mn DESC;
"""

opportunity = pd.read_sql(query, conn)
opportunity

,bank_name,debit_cards,credit_cards,cc_per_100_dc,extra_cards_mn
0,UNION BANK OF INDIA,60574547,551230,0.9,4.55
1,BANK OF BARODA,87335032,2937896,3.4,4.42
2,CANARA BANK,59396108,997943,1.7,4.01
3,BANK OF INDIA,38747535,73696,0.2,3.19
4,PUNJAB NATIONAL BANK,41015342,574560,1.4,2.88
5,INDIAN BANK,33020751,282514,0.9,2.50
6,CENTRAL BANK OF INDIA,28178425,0,0.0,2.37
7,INDIAN OVERSEAS BANK,20478463,65086,0.3,1.66
8,BANK OF MAHARASHTRA,14022969,31278,0.2,1.15
9,UCO BANK,12454245,0,0.0,1.05


Question 4: is ATM cash withdrawal declining?

Step 10: ATM cash withdrawals per day

In [16]:
query = """
WITH monthly AS (
    SELECT
        month,
        SUM(COALESCE(dc_atm_cash_vol, 0) + COALESCE(cc_atm_cash_vol, 0)) AS atm_txns,
        SUM(COALESCE(dc_atm_cash_val_cr, 0) + COALESCE(cc_atm_cash_val_cr, 0)) AS atm_value_cr,
        CAST(strftime('%d', date(month, '+1 month', '-1 day')) AS INTEGER) AS days_in_month
    FROM rbi_bank_cards
    GROUP BY month
)
SELECT
    month,
    days_in_month,
    ROUND(atm_txns / 1000000.0 / days_in_month, 2) AS atm_txns_per_day_mn,
    ROUND(atm_value_cr / days_in_month, 0) AS atm_value_per_day_cr,
    ROUND(atm_value_cr * 10000000.0 / atm_txns, 0) AS avg_withdrawal_rs
FROM monthly
ORDER BY month;
"""

atm = pd.read_sql(query, conn)
atm

,month,days_in_month,atm_txns_per_day_mn,atm_value_per_day_cr,avg_withdrawal_rs
0,2024-01-01,31,17.03,8378.0,4919.0
1,2024-02-01,29,17.75,8835.0,4978.0
2,2024-03-01,31,17.86,8941.0,5005.0
3,2024-04-01,30,17.39,8830.0,5078.0
4,2024-05-01,31,16.59,8362.0,5041.0
5,2024-06-01,30,16.85,8476.0,5031.0
6,2024-07-01,31,16.29,8045.0,4940.0
7,2024-08-01,31,16.57,8198.0,4947.0
8,2024-09-01,30,16.43,8145.0,4956.0
9,2024-10-01,31,17.82,9178.0,5151.0


ATM cash use is declining gradually, not collapsing: withdrawals per day fell about 6% across 2024, confirmed by both RBI and NPCI data, while the value withdrawn fell only about 3% because people now withdraw larger amounts less often. Cash demand still spikes around festivals like Diwali.

Step 11: Is card spending moving online?
Business question 5: Are people spending more with cards online (e-commerce) than in shops (PoS), and does it differ between credit and debit cards?

In [17]:
query = """
SELECT
    month,
    ROUND(SUM(cc_pos_val_cr), 0) AS cc_instore_cr,
    ROUND(SUM(cc_ecom_val_cr), 0) AS cc_online_cr,
    ROUND(SUM(cc_ecom_val_cr) * 100.0
          / (SUM(cc_pos_val_cr) + SUM(cc_ecom_val_cr)), 1) AS cc_online_share_pct,
    ROUND(SUM(dc_pos_val_cr), 0) AS dc_instore_cr,
    ROUND(SUM(dc_ecom_val_cr), 0) AS dc_online_cr,
    ROUND(SUM(dc_ecom_val_cr) * 100.0
          / (SUM(dc_pos_val_cr) + SUM(dc_ecom_val_cr)), 1) AS dc_online_share_pct
FROM rbi_bank_cards
GROUP BY month
ORDER BY month;
"""

card_spend = pd.read_sql(query, conn)
card_spend

,month,cc_instore_cr,cc_online_cr,cc_online_share_pct,dc_instore_cr,dc_online_cr,dc_online_share_pct
0,2024-01-01,58532.0,107912.0,64.8,30383.0,14883.0,32.9
1,2024-02-01,54431.0,94774.0,63.5,27918.0,13324.0,32.3
2,2024-03-01,60378.0,104080.0,63.3,29309.0,15214.0,34.2
3,2024-04-01,61982.0,94516.0,60.4,30022.0,13681.0,31.3
4,2024-05-01,63831.0,101123.0,61.3,29772.0,13588.0,31.3
5,2024-06-01,59417.0,99404.0,62.6,27629.0,13441.0,32.7
6,2024-07-01,62284.0,110386.0,63.9,28600.0,15966.0,35.8
7,2024-08-01,63048.0,105136.0,62.5,29346.0,14010.0,32.3
8,2024-09-01,60857.0,115345.0,65.5,25777.0,14146.0,35.4
9,2024-10-01,79293.0,122495.0,60.7,32091.0,14551.0,31.2


Card spending isn’t shifting online: the online share of credit card spending stayed around 60–65% and dipped slightly, as in-store credit card spending grew faster (about +27% vs +10%). The bigger shift is between card types: debit card spending declined while credit card spending grew, suggesting UPI is replacing debit cards for everyday payments, while credit cards hold their place for credit-based and high-value spending.

In [ ]:
Step 12: Digital infrastructure, UPI QR codes vs card machines

Business question 3: Are banks building infrastructure for UPI faster than for cards, and does it differ by bank type?

In [18]:
query = """
WITH snap AS (
    SELECT
        bank_category,
        SUM(CASE WHEN month = '2024-01-01' THEN pos_terminals END) AS pos_jan,
        SUM(CASE WHEN month = '2024-12-01' THEN pos_terminals END) AS pos_dec,
        SUM(CASE WHEN month = '2024-01-01' THEN upi_qr END) AS qr_jan,
        SUM(CASE WHEN month = '2024-12-01' THEN upi_qr END) AS qr_dec
    FROM rbi_bank_cards
    WHERE month IN ('2024-01-01', '2024-12-01')
    GROUP BY bank_category
),
with_total AS (
    SELECT * FROM snap
    UNION ALL
    SELECT 'All banks', SUM(pos_jan), SUM(pos_dec), SUM(qr_jan), SUM(qr_dec)
    FROM snap
)
SELECT
    bank_category,
    pos_jan,
    pos_dec,
    ROUND((pos_dec - pos_jan) * 100.0 / NULLIF(pos_jan, 0), 1) AS pos_growth_pct,
    qr_jan,
    qr_dec,
    ROUND((qr_dec - qr_jan) * 100.0 / NULLIF(qr_jan, 0), 1) AS qr_growth_pct,
    ROUND(qr_dec * 1.0 / NULLIF(pos_dec, 0), 0) AS qr_per_pos_dec
FROM with_total
ORDER BY qr_dec DESC;
"""

infra = pd.read_sql(query, conn)
infra

,bank_category,pos_jan,pos_dec,pos_growth_pct,qr_jan,qr_dec,qr_growth_pct,qr_per_pos_dec
0,All banks,8593226,10000549,16.4,321379274,633529503,97.1,63.0
1,Private Sector Banks,6238036,8219359,31.8,263587023,608039506,130.7,74.0
2,Public Sector Banks,1549914,1741306,12.3,15616151,20023587,28.2,11.0
3,Payment Banks,725675,0,-100.0,39878024,3952275,-90.1,NaN
4,Small Finance Banks,26417,2793,-89.4,2292843,1506713,-34.3,539.0
5,Foreign Banks,53184,37091,-30.3,5233,7422,41.8,0.0


In [19]:
query = """
WITH snap AS (
    SELECT
        bank_name,
        bank_category,
        SUM(CASE WHEN month = '2024-01-01' THEN upi_qr END) AS qr_jan,
        SUM(CASE WHEN month = '2024-12-01' THEN upi_qr END) AS qr_dec,
        SUM(CASE WHEN month = '2024-01-01' THEN pos_terminals END) AS pos_jan,
        SUM(CASE WHEN month = '2024-12-01' THEN pos_terminals END) AS pos_dec
    FROM rbi_bank_cards
    WHERE month IN ('2024-01-01', '2024-12-01')
    GROUP BY bank_name, bank_category
)
SELECT
    bank_name,
    bank_category,
    qr_jan,
    qr_dec,
    COALESCE(qr_dec, 0) - COALESCE(qr_jan, 0) AS qr_change,
    pos_jan,
    pos_dec,
    COALESCE(pos_dec, 0) - COALESCE(pos_jan, 0) AS pos_change
FROM snap
ORDER BY ABS(COALESCE(qr_dec, 0) - COALESCE(qr_jan, 0)) DESC
LIMIT 10;
"""

qr_movers = pd.read_sql(query, conn)
qr_movers

,bank_name,bank_category,qr_jan,qr_dec,qr_change,pos_jan,pos_dec,pos_change
0,YES BANK LTD,Private Sector Banks,174464847,491987520,317522673,74540,98651,24111
1,PAYTM PAYMENTS BANK,Payment Banks,35743321,0,-35743321,725675,0,-725675
2,AXIS BANK LTD,Private Sector Banks,75279432,96258045,20978613,1665417,1967973,302556
3,INDUSIND BANK LTD,Private Sector Banks,3873664,7134005,3260341,215819,249297,33478
4,CANARA BANK,Public Sector Banks,2030488,3344640,1314152,72047,74536,2489
5,AIRTEL PAYMENTS BANK,Payment Banks,3348120,2157842,-1190278,0,0,0
6,STATE BANK OF INDIA,Public Sector Banks,3496334,4584539,1088205,1282632,1480273,197641
7,INDIA POST PAYMENTS BANK,Payment Banks,776268,1685722,909454,0,0,0
8,FEDERAL BANK LTD,Private Sector Banks,4849424,5737909,888485,19743,21058,1315
9,HDFC BANK LTD,Private Sector Banks,0,814156,814156,1570324,1695039,124715



bank_name	bank_category	qr_jan	qr_dec	qr_change	pos_jan	pos_dec	pos_change
0	YES BANK LTD	Private Sector Banks	174464847	491987520	317522673	74540	98651	24111
1	PAYTM PAYMENTS BANK	Payment Banks	35743321	0	-35743321	725675	0	-725675
2	AXIS BANK LTD	Private Sector Banks	75279432	96258045	20978613	1665417	1967973	302556
3	INDUSIND BANK LTD	Private Sector Banks	3873664	7134005	3260341	215819	249297	33478
4	CANARA BANK	Public Sector Banks	2030488	3344640	1314152	72047	74536	2489
5	AIRTEL PAYMENTS BANK	Payment Banks	3348120	2157842	-1190278	0	0	0
6	STATE BANK OF INDIA	Public Sector Banks	3496334	4584539	1088205	1282632	1480273	197641
7	INDIA POST PAYMENTS BANK	Payment Banks	776268	1685722	909454	0	0	0
8	FEDERAL BANK LTD	Private Sector Banks	4849424	5737909	888485	19743	21058	1315
9	HDFC BANK LTD	Private Sector Banks	0	814156	814156	1570324	1695039	124715


Step 14: One quick follow-up, the small finance bank PoS drop

In [20]:
query = """
SELECT
    bank_name,
    SUM(CASE WHEN month = '2024-01-01' THEN pos_terminals END) AS pos_jan,
    SUM(CASE WHEN month = '2024-12-01' THEN pos_terminals END) AS pos_dec
FROM rbi_bank_cards
WHERE bank_category = 'Small Finance Banks'
  AND month IN ('2024-01-01', '2024-12-01')
GROUP BY bank_name
ORDER BY pos_jan DESC;
"""

pd.read_sql(query, conn)

,bank_name,pos_jan,pos_dec
0,EQUITAS SMALL FINANCE BANK LTD,26394,2793.0
1,UNITY SMALL FINANCE BANK LTD,23,0.0
2,UTKARSH SMALL FINANCE BANK LTD,0,0.0
3,UJJIVAN SMALL FINANCE BANK LTD,0,0.0
4,SURYODAY SMALL FINANCE BANK LTD,0,0.0
5,SHIVALIK SMALL FINANCE BANK LTD,0,0.0
6,NORTH EAST SMALL FINANCE BANK LTD,0,0.0
7,JANA SMALL FINANCE BANK LTD,0,0.0
8,FINCARE SMALL FINANCE BANK LTD,0,NaN
9,ESAF SMALL FINANCE BANK LTD,0,0.0


In [21]:
sql_dir = project / 'sql'
sql_dir.mkdir(exist_ok=True)

files = {
'01_payment_systems.sql': """
-- Q1: Is UPI replacing other payment methods, or growing alongside them?
-- Source: npci_monthly (NPCI, Jan-Dec 2024)
-- Method: average daily volume, Q1 vs Q4 (removes month-length effects and single-month noise)

-- Month-over-month growth in daily volume: UPI vs ATM network (NFS)
WITH monthly AS (
    SELECT
        month,
        SUM(CASE WHEN system = 'UPI' THEN avg_daily_volume_mn END) AS upi_daily_mn,
        SUM(CASE WHEN system = 'NFS' THEN avg_daily_volume_mn END) AS atm_daily_mn
    FROM npci_monthly
    GROUP BY month
)
SELECT
    month,
    upi_daily_mn,
    ROUND((upi_daily_mn - LAG(upi_daily_mn) OVER (ORDER BY month)) * 100.0
          / LAG(upi_daily_mn) OVER (ORDER BY month), 1) AS upi_daily_mom_pct,
    atm_daily_mn,
    ROUND((atm_daily_mn - LAG(atm_daily_mn) OVER (ORDER BY month)) * 100.0
          / LAG(atm_daily_mn) OVER (ORDER BY month), 1) AS atm_daily_mom_pct
FROM monthly
ORDER BY month;

-- Q1 vs Q4 growth for every payment system
WITH quarterly AS (
    SELECT
        system,
        CASE
            WHEN CAST(strftime('%m', month) AS INTEGER) <= 3  THEN 'Q1'
            WHEN CAST(strftime('%m', month) AS INTEGER) >= 10 THEN 'Q4'
        END AS quarter,
        avg_daily_volume_mn
    FROM npci_monthly
)
SELECT
    system,
    ROUND(AVG(CASE WHEN quarter = 'Q1' THEN avg_daily_volume_mn END), 2) AS q1_daily_mn,
    ROUND(AVG(CASE WHEN quarter = 'Q4' THEN avg_daily_volume_mn END), 2) AS q4_daily_mn,
    ROUND((AVG(CASE WHEN quarter = 'Q4' THEN avg_daily_volume_mn END)
         - AVG(CASE WHEN quarter = 'Q1' THEN avg_daily_volume_mn END)) * 100.0
         / AVG(CASE WHEN quarter = 'Q1' THEN avg_daily_volume_mn END), 1) AS q1_to_q4_pct
FROM quarterly
GROUP BY system
ORDER BY q1_to_q4_pct DESC;
""",

'02_credit_cards.sql': """
-- Q2: Which banks dominate credit cards, and where is the opportunity?
-- Source: rbi_bank_cards (RBI, Dec 2024 month-end stock)

-- Top 10 credit card issuers with market share and rank
SELECT
    bank_name,
    bank_category,
    credit_cards,
    ROUND(credit_cards * 100.0 / SUM(credit_cards) OVER (), 2) AS market_share_pct,
    RANK() OVER (ORDER BY credit_cards DESC) AS card_rank
FROM rbi_bank_cards
WHERE month = '2024-12-01'
ORDER BY credit_cards DESC
LIMIT 10;

-- Credit vs debit card share by bank category
SELECT
    bank_category,
    COUNT(*) AS num_banks,
    SUM(credit_cards) AS credit_cards,
    ROUND(SUM(credit_cards) * 100.0 / SUM(SUM(credit_cards)) OVER (), 1) AS credit_share_pct,
    SUM(debit_cards) AS debit_cards,
    ROUND(SUM(debit_cards) * 100.0 / SUM(SUM(debit_cards)) OVER (), 1) AS debit_share_pct
FROM rbi_bank_cards
WHERE month = '2024-12-01'
GROUP BY bank_category
ORDER BY credit_cards DESC;

-- Credit card penetration (credit cards per 100 debit cards), banks with 10 Mn+ debit cards
SELECT
    bank_name,
    bank_category,
    debit_cards,
    credit_cards,
    ROUND(credit_cards * 100.0 / debit_cards, 1) AS cc_per_100_dc
FROM rbi_bank_cards
WHERE month = '2024-12-01'
  AND debit_cards >= 10000000
ORDER BY cc_per_100_dc ASC;

-- Opportunity sizing: extra credit cards if each public sector bank matched SBI's penetration
WITH sbi AS (
    SELECT credit_cards * 1.0 / debit_cards AS benchmark_ratio
    FROM rbi_bank_cards
    WHERE month = '2024-12-01'
      AND bank_name = 'STATE BANK OF INDIA'
)
SELECT
    r.bank_name,
    r.debit_cards,
    r.credit_cards,
    ROUND(r.credit_cards * 100.0 / r.debit_cards, 1) AS cc_per_100_dc,
    ROUND((r.debit_cards * s.benchmark_ratio - r.credit_cards) / 1000000.0, 2) AS extra_cards_mn
FROM rbi_bank_cards AS r
CROSS JOIN sbi AS s
WHERE r.month = '2024-12-01'
  AND r.bank_category = 'Public Sector Banks'
  AND r.bank_name <> 'STATE BANK OF INDIA'
ORDER BY extra_cards_mn DESC;
""",

'03_infrastructure.sql': """
-- Q3: Are banks building UPI acceptance faster than card acceptance?
-- Source: rbi_bank_cards (RBI, Jan vs Dec 2024 month-end stock)

-- UPI QR codes vs PoS terminals by bank category, with an All banks total
WITH snap AS (
    SELECT
        bank_category,
        SUM(CASE WHEN month = '2024-01-01' THEN pos_terminals END) AS pos_jan,
        SUM(CASE WHEN month = '2024-12-01' THEN pos_terminals END) AS pos_dec,
        SUM(CASE WHEN month = '2024-01-01' THEN upi_qr END) AS qr_jan,
        SUM(CASE WHEN month = '2024-12-01' THEN upi_qr END) AS qr_dec
    FROM rbi_bank_cards
    WHERE month IN ('2024-01-01', '2024-12-01')
    GROUP BY bank_category
),
with_total AS (
    SELECT * FROM snap
    UNION ALL
    SELECT 'All banks', SUM(pos_jan), SUM(pos_dec), SUM(qr_jan), SUM(qr_dec)
    FROM snap
)
SELECT
    bank_category,
    pos_jan,
    pos_dec,
    ROUND((pos_dec - pos_jan) * 100.0 / NULLIF(pos_jan, 0), 1) AS pos_growth_pct,
    qr_jan,
    qr_dec,
    ROUND((qr_dec - qr_jan) * 100.0 / NULLIF(qr_jan, 0), 1) AS qr_growth_pct,
    ROUND(qr_dec * 1.0 / NULLIF(pos_dec, 0), 0) AS qr_per_pos_dec
FROM with_total
ORDER BY qr_dec DESC;

-- Banks with the largest change in UPI QR codes (up or down)
WITH snap AS (
    SELECT
        bank_name,
        bank_category,
        SUM(CASE WHEN month = '2024-01-01' THEN upi_qr END) AS qr_jan,
        SUM(CASE WHEN month = '2024-12-01' THEN upi_qr END) AS qr_dec,
        SUM(CASE WHEN month = '2024-01-01' THEN pos_terminals END) AS pos_jan,
        SUM(CASE WHEN month = '2024-12-01' THEN pos_terminals END) AS pos_dec
    FROM rbi_bank_cards
    WHERE month IN ('2024-01-01', '2024-12-01')
    GROUP BY bank_name, bank_category
)
SELECT
    bank_name,
    bank_category,
    qr_jan,
    qr_dec,
    COALESCE(qr_dec, 0) - COALESCE(qr_jan, 0) AS qr_change,
    pos_jan,
    pos_dec,
    COALESCE(pos_dec, 0) - COALESCE(pos_jan, 0) AS pos_change
FROM snap
ORDER BY ABS(COALESCE(qr_dec, 0) - COALESCE(qr_jan, 0)) DESC
LIMIT 10;
""",

'04_atm_cash.sql': """
-- Q4: Is ATM cash withdrawal declining?
-- Source: rbi_bank_cards (RBI, all ATM withdrawals by debit and credit cards, Jan-Dec 2024)
-- Method: per-day figures to remove month-length effects

WITH monthly AS (
    SELECT
        month,
        SUM(COALESCE(dc_atm_cash_vol, 0) + COALESCE(cc_atm_cash_vol, 0)) AS atm_txns,
        SUM(COALESCE(dc_atm_cash_val_cr, 0) + COALESCE(cc_atm_cash_val_cr, 0)) AS atm_value_cr,
        CAST(strftime('%d', date(month, '+1 month', '-1 day')) AS INTEGER) AS days_in_month
    FROM rbi_bank_cards
    GROUP BY month
)
SELECT
    month,
    days_in_month,
    ROUND(atm_txns / 1000000.0 / days_in_month, 2) AS atm_txns_per_day_mn,
    ROUND(atm_value_cr / days_in_month, 0) AS atm_value_per_day_cr,
    ROUND(atm_value_cr * 10000000.0 / atm_txns, 0) AS avg_withdrawal_rs
FROM monthly
ORDER BY month;
""",

'05_card_spending.sql': """
-- Q5: Is card spending moving online?
-- Source: rbi_bank_cards (RBI, card payment values, Jan-Dec 2024)
-- Note: 'others' channel excluded (small, not clearly in-store or online)

SELECT
    month,
    ROUND(SUM(cc_pos_val_cr), 0) AS cc_instore_cr,
    ROUND(SUM(cc_ecom_val_cr), 0) AS cc_online_cr,
    ROUND(SUM(cc_ecom_val_cr) * 100.0
          / (SUM(cc_pos_val_cr) + SUM(cc_ecom_val_cr)), 1) AS cc_online_share_pct,
    ROUND(SUM(dc_pos_val_cr), 0) AS dc_instore_cr,
    ROUND(SUM(dc_ecom_val_cr), 0) AS dc_online_cr,
    ROUND(SUM(dc_ecom_val_cr) * 100.0
          / (SUM(dc_pos_val_cr) + SUM(dc_ecom_val_cr)), 1) AS dc_online_share_pct
FROM rbi_bank_cards
GROUP BY month
ORDER BY month;
""",
}

for name, sql in files.items():
    (sql_dir / name).write_text(sql.strip() + '\n', encoding='utf-8')
    print("Saved", name)

Saved 01_payment_systems.sql
Saved 02_credit_cards.sql
Saved 03_infrastructure.sql
Saved 04_atm_cash.sql
Saved 05_card_spending.sql
